In [193]:
# Step 2: Import Python Packages: Pandas, Numpy, and Statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
import os

In [194]:
# Step 3: Set I/O Folders
InputPath = 'inputFolder'

os.makedirs('outputFolder', exist_ok=True)
OutputPath = 'outputFolder'

In [195]:
df = pd.read_csv(InputPath + '/Case Study 01_PostTrade.csv')
df.head()

,Broker,TradeDate,Symbol,OrderSide,ADV,Volatility,Beta,MktCap_MM,OrderShares,ExecutedShares,POV,Pavg,Pd,P0,Pn,VWAP
0,Broker A,8/5/2024,AAPL,Buy,72106864.0,0.68,1.041596,2901664,2294800,2294800,0.153908,197.52,193.27,193.49,199.25,197.379224
1,Broker C,8/5/2024,MSFT,Buy,26193634.0,0.40,0.991805,2669692,658600,658600,0.151336,367.34,362.82,362.88,370.55,367.287875
2,Broker B,8/5/2024,NVDA,Sell,323825056.0,0.81,1.904370,2364604,14230300,14230300,0.148172,96.08,98.77,98.63,97.66,96.097986
3,Broker A,8/5/2024,AMZN,Sell,56974192.0,0.58,1.184176,1775661,519200,519200,0.091066,167.46,169.60,169.56,168.62,167.565463
4,Broker A,8/5/2024,META,Sell,20542520.0,0.67,1.181923,1223500,526900,526900,0.173758,481.83,491.33,491.27,489.22,481.921026


In [ ]:
# Step 4: Set MI Parameters - Calculated from Non-Linear Regression Analysis
a1 = 883.58
a2 = 0.35
a3 = 0.75
a4 = 0.82
b1 = 0.96
df['Alpha_bp'] = 0

In [197]:
df['OrderSide'] = df['OrderSide'].map({'Buy': 1, 'Sell': -1}) # Map side to value 1 or -1
df['Size_fraction'] = df['OrderShares'] / df['ADV']

## Implementation Shortfall

In [198]:
COMMISSION_PER_SHARE = 0.01

df['Fixed_Cost'] = df['ExecutedShares'] * COMMISSION_PER_SHARE
df['Delay_Cost'] = df['OrderShares'] * (df['P0'] - df['Pd'])
df['Execution_Cost'] =  df['ExecutedShares'] * (df['Pavg'] - df['P0'])
df['Opportunity_Cost'] = (df['OrderShares'] - df['ExecutedShares']) * (df['Pn'] - df['P0'])
df['IS_Total'] = df['OrderSide'] * (df['Delay_Cost'] + df['Execution_Cost'] + df['Opportunity_Cost']) + df['Fixed_Cost']

In [199]:
df

,Broker,TradeDate,Symbol,OrderSide,ADV,Volatility,Beta,MktCap_MM,OrderShares,ExecutedShares,...,P0,Pn,VWAP,Alpha_bp,Size_fraction,Fixed_Cost,Delay_Cost,Execution_Cost,Opportunity_Cost,IS_Total
0,Broker A,8/5/2024,AAPL,1,7.210686e+07,0.68,1.041596,2901664,2294800,2294800,...,193.49,199.25,197.379224,0,0.031825,22948.0,504856.0,9248044.0,0.0,9775848.0
1,Broker C,8/5/2024,MSFT,1,2.619363e+07,0.40,0.991805,2669692,658600,658600,...,362.88,370.55,367.287875,0,0.025144,6586.0,39516.0,2937356.0,0.0,2983458.0
2,Broker B,8/5/2024,NVDA,-1,3.238251e+08,0.81,1.904370,2364604,14230300,14230300,...,98.63,97.66,96.097986,0,0.043944,142303.0,-1992242.0,-36287265.0,-0.0,38421810.0
3,Broker A,8/5/2024,AMZN,-1,5.697419e+07,0.58,1.184176,1775661,519200,519200,...,169.56,168.62,167.565463,0,0.009113,5192.0,-20768.0,-1090320.0,-0.0,1116280.0
4,Broker A,8/5/2024,META,-1,2.054252e+07,0.67,1.181923,1223500,526900,526900,...,491.27,489.22,481.921026,0,0.025649,5269.0,-31614.0,-4973936.0,-0.0,5010819.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5005,Broker C,8/9/2024,BEN,1,5.018205e+06,0.50,0.993405,9116,77100,77100,...,17.43,17.79,17.645245,0,0.015364,771.0,6168.0,16191.0,0.0,23130.0
5006,Broker E,8/9/2024,CRL,1,1.923912e+06,1.15,0.819497,5043,52800,52800,...,102.96,107.47,105.500238,0,0.027444,528.0,15312.0,140976.0,0.0,156816.0
5007,Broker D,8/9/2024,CZR,1,6.304107e+06,0.76,1.529065,5300,164200,164200,...,25.03,25.73,25.538963,0,0.026047,1642.0,4926.0,80458.0,0.0,87026.0
5008,Broker A,8/9/2024,MHK,-1,8.187285e+05,0.53,1.147716,6250,34800,34800,...,99.81,98.72,98.005129,0,0.042505,348.0,-5568.0,-66816.0,-0.0,72732.0


## Arrival Cost

In [200]:
df['Arrival_Cost_bp'] = df['OrderSide'] * ((df['Pavg'] - df['P0']) / df['P0']) * 10000

df

,Broker,TradeDate,Symbol,OrderSide,ADV,Volatility,Beta,MktCap_MM,OrderShares,ExecutedShares,...,Pn,VWAP,Alpha_bp,Size_fraction,Fixed_Cost,Delay_Cost,Execution_Cost,Opportunity_Cost,IS_Total,Arrival_Cost_bp
0,Broker A,8/5/2024,AAPL,1,7.210686e+07,0.68,1.041596,2901664,2294800,2294800,...,199.25,197.379224,0,0.031825,22948.0,504856.0,9248044.0,0.0,9775848.0,208.279498
1,Broker C,8/5/2024,MSFT,1,2.619363e+07,0.40,0.991805,2669692,658600,658600,...,370.55,367.287875,0,0.025144,6586.0,39516.0,2937356.0,0.0,2983458.0,122.905644
2,Broker B,8/5/2024,NVDA,-1,3.238251e+08,0.81,1.904370,2364604,14230300,14230300,...,97.66,96.097986,0,0.043944,142303.0,-1992242.0,-36287265.0,-0.0,38421810.0,258.542026
3,Broker A,8/5/2024,AMZN,-1,5.697419e+07,0.58,1.184176,1775661,519200,519200,...,168.62,167.565463,0,0.009113,5192.0,-20768.0,-1090320.0,-0.0,1116280.0,123.849965
4,Broker A,8/5/2024,META,-1,2.054252e+07,0.67,1.181923,1223500,526900,526900,...,489.22,481.921026,0,0.025649,5269.0,-31614.0,-4973936.0,-0.0,5010819.0,192.155027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5005,Broker C,8/9/2024,BEN,1,5.018205e+06,0.50,0.993405,9116,77100,77100,...,17.79,17.645245,0,0.015364,771.0,6168.0,16191.0,0.0,23130.0,120.481928
5006,Broker E,8/9/2024,CRL,1,1.923912e+06,1.15,0.819497,5043,52800,52800,...,107.47,105.500238,0,0.027444,528.0,15312.0,140976.0,0.0,156816.0,259.324009
5007,Broker D,8/9/2024,CZR,1,6.304107e+06,0.76,1.529065,5300,164200,164200,...,25.73,25.538963,0,0.026047,1642.0,4926.0,80458.0,0.0,87026.0,195.765082
5008,Broker A,8/9/2024,MHK,-1,8.187285e+05,0.53,1.147716,6250,34800,34800,...,98.72,98.005129,0,0.042505,348.0,-5568.0,-66816.0,-0.0,72732.0,192.365494


## VWAP Slippage

In [201]:
df['VWAP_Slippage_bp'] = df['OrderSide'] * ((df['Pavg'] - df['VWAP']) / df['VWAP']) * 10000

df

,Broker,TradeDate,Symbol,OrderSide,ADV,Volatility,Beta,MktCap_MM,OrderShares,ExecutedShares,...,VWAP,Alpha_bp,Size_fraction,Fixed_Cost,Delay_Cost,Execution_Cost,Opportunity_Cost,IS_Total,Arrival_Cost_bp,VWAP_Slippage_bp
0,Broker A,8/5/2024,AAPL,1,7.210686e+07,0.68,1.041596,2901664,2294800,2294800,...,197.379224,0,0.031825,22948.0,504856.0,9248044.0,0.0,9775848.0,208.279498,7.132270
1,Broker C,8/5/2024,MSFT,1,2.619363e+07,0.40,0.991805,2669692,658600,658600,...,367.287875,0,0.025144,6586.0,39516.0,2937356.0,0.0,2983458.0,122.905644,1.419187
2,Broker B,8/5/2024,NVDA,-1,3.238251e+08,0.81,1.904370,2364604,14230300,14230300,...,96.097986,0,0.043944,142303.0,-1992242.0,-36287265.0,-0.0,38421810.0,258.542026,1.871679
3,Broker A,8/5/2024,AMZN,-1,5.697419e+07,0.58,1.184176,1775661,519200,519200,...,167.565463,0,0.009113,5192.0,-20768.0,-1090320.0,-0.0,1116280.0,123.849965,6.293863
4,Broker A,8/5/2024,META,-1,2.054252e+07,0.67,1.181923,1223500,526900,526900,...,481.921026,0,0.025649,5269.0,-31614.0,-4973936.0,-0.0,5010819.0,192.155027,1.888812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5005,Broker C,8/9/2024,BEN,1,5.018205e+06,0.50,0.993405,9116,77100,77100,...,17.645245,0,0.015364,771.0,6168.0,16191.0,0.0,23130.0,120.481928,-2.972489
5006,Broker E,8/9/2024,CRL,1,1.923912e+06,1.15,0.819497,5043,52800,52800,...,105.500238,0,0.027444,528.0,15312.0,140976.0,0.0,156816.0,259.324009,12.299640
5007,Broker D,8/9/2024,CZR,1,6.304107e+06,0.76,1.529065,5300,164200,164200,...,25.538963,0,0.026047,1642.0,4926.0,80458.0,0.0,87026.0,195.765082,-7.425309
5008,Broker A,8/9/2024,MHK,-1,8.187285e+05,0.53,1.147716,6250,34800,34800,...,98.005129,0,0.042505,348.0,-5568.0,-66816.0,-0.0,72732.0,192.365494,11.747223


## Value-Add

In [202]:
# ------------ Calculate MI, PA, TR ------------
df['I_star'] = a1 * (df['Size_fraction']**a2) * (df['Volatility']**a3)
# Market Impact (MI) = I* * (b1 * POV^a4 + (1 - b1))
df['MI_bp'] = df['I_star'] * (b1 * (df['POV']**a4) + (1 - b1))
# Timing Risk (TR) = sigma * sqrt(1/3 * 1/250 * Size * (1-POV)/POV) * 10000
df['TR_bp'] = df['Volatility'] * np.sqrt((1/3) * (1/250) * df['Size_fraction'] * (1-df['POV'])/df['POV']) * 10000
# Price Appreciation (Alpha_bp = 0)
df['PA_bp'] = df['OrderSide'] * (1/2) * df['Alpha_bp'] * df['Size_fraction'] * ((1 - df['POV'])/ df['POV'])


df['Expected_Cost'] = df['MI_bp']
df['Value_Add_bp'] = df['MI_bp'] - df['Arrival_Cost_bp']

df

,Broker,TradeDate,Symbol,OrderSide,ADV,Volatility,Beta,MktCap_MM,OrderShares,ExecutedShares,...,Opportunity_Cost,IS_Total,Arrival_Cost_bp,VWAP_Slippage_bp,I_star,MI_bp,TR_bp,PA_bp,Expected_Cost,Value_Add_bp
0,Broker A,8/5/2024,AAPL,1,7.210686e+07,0.68,1.041596,2901664,2294800,2294800,...,0.0,9775848.0,208.279498,7.132270,100.190620,40.523625,103.858032,0.0,40.523625,-167.755873
1,Broker C,8/5/2024,MSFT,1,2.619363e+07,0.40,0.991805,2669692,658600,658600,...,0.0,2983458.0,122.905644,1.419187,59.816258,24.000572,54.845367,0.0,24.000572,-98.905072
2,Broker B,8/5/2024,NVDA,-1,3.238251e+08,0.81,1.904370,2364604,14230300,14230300,...,-0.0,38421810.0,258.542026,1.871679,134.238420,53.323789,148.661965,-0.0,53.323789,-205.218236
3,Broker A,8/5/2024,AMZN,-1,5.697419e+07,0.58,1.184176,1775661,519200,519200,...,-0.0,1116280.0,123.849965,6.293863,47.583952,15.023950,63.872550,-0.0,15.023950,-108.826015
4,Broker A,8/5/2024,META,-1,2.054252e+07,0.67,1.181923,1223500,526900,526900,...,-0.0,5010819.0,192.155027,1.888812,88.951772,38.116379,85.440254,-0.0,38.116379,-154.038647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5005,Broker C,8/9/2024,BEN,1,5.018205e+06,0.50,0.993405,9116,77100,77100,...,0.0,23130.0,120.481928,-2.972489,55.276655,17.852107,69.618071,0.0,17.852107,-102.629821
5006,Broker E,8/9/2024,CRL,1,1.923912e+06,1.15,0.819497,5043,52800,52800,...,0.0,156816.0,259.324009,12.299640,137.977723,48.142203,195.247556,0.0,48.142203,-211.181807
5007,Broker D,8/9/2024,CZR,1,6.304107e+06,0.76,1.529065,5300,164200,164200,...,0.0,87026.0,195.765082,-7.425309,98.524955,37.760900,112.219439,0.0,37.760900,-158.004182
5008,Broker A,8/9/2024,MHK,-1,8.187285e+05,0.53,1.147716,6250,34800,34800,...,-0.0,72732.0,192.365494,11.747223,96.047891,43.758962,80.410411,-0.0,43.758962,-148.606532


## Z-Score

In [203]:
df['Z_Score'] = df['Value_Add_bp'] / df['TR_bp']

df

,Broker,TradeDate,Symbol,OrderSide,ADV,Volatility,Beta,MktCap_MM,OrderShares,ExecutedShares,...,IS_Total,Arrival_Cost_bp,VWAP_Slippage_bp,I_star,MI_bp,TR_bp,PA_bp,Expected_Cost,Value_Add_bp,Z_Score
0,Broker A,8/5/2024,AAPL,1,7.210686e+07,0.68,1.041596,2901664,2294800,2294800,...,9775848.0,208.279498,7.132270,100.190620,40.523625,103.858032,0.0,40.523625,-167.755873,-1.615242
1,Broker C,8/5/2024,MSFT,1,2.619363e+07,0.40,0.991805,2669692,658600,658600,...,2983458.0,122.905644,1.419187,59.816258,24.000572,54.845367,0.0,24.000572,-98.905072,-1.803344
2,Broker B,8/5/2024,NVDA,-1,3.238251e+08,0.81,1.904370,2364604,14230300,14230300,...,38421810.0,258.542026,1.871679,134.238420,53.323789,148.661965,-0.0,53.323789,-205.218236,-1.380435
3,Broker A,8/5/2024,AMZN,-1,5.697419e+07,0.58,1.184176,1775661,519200,519200,...,1116280.0,123.849965,6.293863,47.583952,15.023950,63.872550,-0.0,15.023950,-108.826015,-1.703799
4,Broker A,8/5/2024,META,-1,2.054252e+07,0.67,1.181923,1223500,526900,526900,...,5010819.0,192.155027,1.888812,88.951772,38.116379,85.440254,-0.0,38.116379,-154.038647,-1.802881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5005,Broker C,8/9/2024,BEN,1,5.018205e+06,0.50,0.993405,9116,77100,77100,...,23130.0,120.481928,-2.972489,55.276655,17.852107,69.618071,0.0,17.852107,-102.629821,-1.474184
5006,Broker E,8/9/2024,CRL,1,1.923912e+06,1.15,0.819497,5043,52800,52800,...,156816.0,259.324009,12.299640,137.977723,48.142203,195.247556,0.0,48.142203,-211.181807,-1.081611
5007,Broker D,8/9/2024,CZR,1,6.304107e+06,0.76,1.529065,5300,164200,164200,...,87026.0,195.765082,-7.425309,98.524955,37.760900,112.219439,0.0,37.760900,-158.004182,-1.407993
5008,Broker A,8/9/2024,MHK,-1,8.187285e+05,0.53,1.147716,6250,34800,34800,...,72732.0,192.365494,11.747223,96.047891,43.758962,80.410411,-0.0,43.758962,-148.606532,-1.848101


## Save Broker Report Card to CSV

In [204]:
def get_report_card(data):
    # Grouping results by Broker
    report = data.groupby('Broker').agg(
        Orders=('OrderShares', 'count'),
        Avg_Size=('Size_fraction', 'mean'),
        IS_Total=('IS_Total', 'sum'),
        Delay_Cost=('Delay_Cost', 'sum'),
        Execution_Cost=('Execution_Cost', 'sum'),
        Opportunity_Cost=('Opportunity_Cost', 'sum'),
        Fixed_Cost=('Fixed_Cost', 'sum'),
        Arrival_Cost_bp=('Arrival_Cost_bp', 'mean'),
        VWAP_Slip_bp=('VWAP_Slippage_bp', 'mean'),
        Value_Add_bp=('Value_Add_bp', 'mean'),
        Z_Score=('Z_Score', 'mean')
    )

    # Calculate Total Portfolio Row
    all_brokers = pd.DataFrame({
        'Orders': [data.shape[0]],
        'Avg_Size': [data['Size_fraction'].mean()],
        'IS_Total': [data['IS_Total'].sum()],
        'Delay_Cost': [data['Delay_Cost'].sum()],
        'Execution_Cost': [data['Execution_Cost'].sum()],
        'Opportunity_Cost': [data['Opportunity_Cost'].sum()],
        'Fixed_Cost': [data['Fixed_Cost'].sum()],
        'Arrival_Cost_bp': [data['Arrival_Cost_bp'].mean()],
        'VWAP_Slip_bp': [data['VWAP_Slippage_bp'].mean()],
        'Value_Add_bp': [data['Value_Add_bp'].mean()],
        'Z_Score': [data['Z_Score'].mean()]
    }, index=['All Brokers'])


    return pd.concat([ all_brokers, report])

# --- GENERATING THE TABLES ---

# All Orders
all_orders_rpt = get_report_card(df)

small_orders_rpt = get_report_card(df[df['Size_fraction'] <= 0.05])

large_orders_rpt = get_report_card(df[df['Size_fraction'] > 0.05])

# All Orders report to csv
all_orders_rpt.to_csv(OutputPath + '/CaseStudy01_Results.csv')

# Display All Orders Report
print(all_orders_rpt.round())


             Orders  Avg_Size      IS_Total  Delay_Cost  Execution_Cost  \
All Brokers    5010       0.0  4.103915e+09  45115373.0    1.188137e+09   
Broker A       1005       0.0  6.992095e+08   5928494.0    1.899685e+08   
Broker B       1026       0.0  1.040734e+09   -274838.0    1.313485e+07   
Broker C        966       0.0  7.877930e+08   7861148.0    2.511913e+08   
Broker D       1001       0.0  6.390631e+08  15877479.0    2.308017e+08   
Broker E       1012       0.0  9.371153e+08  15723090.0    5.030407e+08   

             Opportunity_Cost  Fixed_Cost  Arrival_Cost_bp  VWAP_Slip_bp  \
All Brokers        73670260.0  17179037.0            161.0           9.0   
Broker A           10416115.0   3412434.0            163.0          14.0   
Broker B            6422002.0   4109488.0            164.0           7.0   
Broker C           26733787.0   3136177.0            164.0           2.0   
Broker D           20211588.0   2780815.0            157.0           7.0   
Broker E          

## Answer to Questions

1) What brokers provided the best performance?

    Brokers E and D provided the best performance. Broker E had the highest Value Add at -124.95 bp and the best z-score of -1.295. This represents a better performance compared to the average of -129.357 value add and -1.429 z-score.

    Broker D also outperformed the average with a value-add of 124.95 and a z-score of -1.2945

2. What brokers underperformed expectations?

    Brokers whose Value-Add and Z-Score are more negative than the all-broker averages (–129.36 bp and –1.429, respectively) are considered underperformers, as their actual execution costs exceeded what the market impact model predicted.

    Broker B was the worst performer overall, posting a Value-Add of –133.24 bp and a Z-Score of –1.554 — the most negative figures across all brokers and well below the portfolio average. This indicates Broker B's execution costs were substantially higher than expected given the size and risk of the orders they handled.

    Broker C was the second weakest performer, with a Value-Add of –132.13 bp and a Z-Score of –1.446, both below the all-broker average, suggesting consistent but less severe underperformance.
    
    Broker A also underperformed expectations, with a Value-Add of –130.66 bp and a Z-Score of –1.482. While closer to the average than Brokers B and C, Broker A still exceeded expected costs on a risk-adjusted basis.